# Dynamic model on farmland investement and financial management

The model we will be solving is:

$$
V_t(R,P,L,W)= \max_{(x,x_m)}\{E[V_{t+1}(R_{t+1}, P_{t+1}, L_{t+1}, W_{t+1})| R_t=r, P_t=p, L_t=L, W_t=W, x_t=x, x_{mt}=x_m]  \}
$$

Where the Bellman equation is subject to:
$$
\begin{aligned}
\ln R_{t+1} &= \beta_0 + \beta_1 \ln R_t + \varepsilon_{1,t+1} \\[6pt]
\ln P_{t+1} &= \alpha_0 + \alpha_1 \ln P_t + \alpha_2 \ln R_t + \varepsilon_{2,t+1} \\[6pt]
\ln M_{t+1} &= \gamma_0 + \varepsilon_{3,t+1} \\[6pt]
L_{t+1} &= L_t + x_t \\[6pt]
W_{t+1} &= (1 + r)\Big[ W_t - (P_t + \kappa - tc_s)L_t \\
&\quad - (P_t + \kappa + tc)x_t - cL_{t+1} - x_{mt} \Big] \\
&\quad + M_{t+1}x_{mt} + R_{t+1}L_{t+1} + (P_{t+1} + \kappa - tc_s)L_{t+1} \\[6pt]
x_t &\in X_t \\[6pt]
x_{mt} &\in X_{mt} \\[6pt]
r &=
\begin{cases}
r_b & \text{if } W_t - (P_t + \kappa - tc_s)L_t - (P_t + \kappa + tc)x_t - cL_{t+1} - x_{mt} < 0 \\
r_l & \text{otherwise}
\end{cases} \\[10pt]
tc &=
\begin{cases}
tc_p & \text{if } x_t > 0 \\
- tc_s & \text{otherwise}
\end{cases}
\end{aligned}
$$



Hence the state variables are: $$s_t=R_t, P_t, L_t, W_t$$ and the choice is choose a level invested in risky farmland and in the mutual fund; $$d_t= \{x_t, x_{mt} \}$$ 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [18]:
class DynamicModel:
    def __init__(self, beta0=1.511810, beta1=0.742391,
                 alpha0=0.215001, alpha1=0.908361, alpha2=0.079432,
                 gamma0=0.057757, kappa=300, c=231, tc_p=0.01, tc_s=0.06,
                 r_b=0.06, r_l=0.03):
        self.beta0 = beta0
        self.beta1 = beta1

        self.alpha0 = alpha0
        self.alpha1 = alpha1
        self.alpha2 = alpha2
        self.gamma0 = gamma0
        self.kappa = kappa
        self.c = c
        self.tc_p = tc_p
        self.tc_s = tc_s
        self.r_b = r_b
        self.r_l = r_l

    def transaction_cost(self, x):
            if x > 0:
                return self.tc_p
            else:
                return -self.tc_s
            
    def interest_rate(self, cash_after_decisions):
        return self.r_b if cash_after_decisions < 0 else self.r_l
    
    
    def transition(self, R, P, L, W, x, xm, eps1, eps2, eps3):
        # stochastic processes
        lnR_next = self.beta0 + self.beta1 * np.log(R) + eps1
        R_next = np.exp(lnR_next)

        lnP_next = self.alpha0 + self.alpha1 * np.log(P) + self.alpha2 * np.log(R) + eps2
        P_next = np.exp(lnP_next)

        lnM_next = self.gamma0 + eps3
        M_next = np.exp(lnM_next)

        # land next period
        L_next = L + x

        # current cash balance A_t recovered from W_t = A_t + (P_t + kappa - tc_s)L_t
        A = W - (P + self.kappa - self.tc_s) * L

        # transaction cost per acre on the adjustment x
        tc = self.transaction_cost(x)

        # cash after land, production, and mutual fund decisions
        cash_after_decisions = A - (P + self.kappa + tc) * x - self.c * L_next - xm

        r = self.interest_rate(cash_after_decisions)

        # next cash balance
        A_next = (1 + r) * cash_after_decisions + M_next * xm + R_next * L_next

        # next wealth
        W_next = A_next + (P_next + self.kappa - self.tc_s) * L_next

        return R_next, P_next, L_next, W_next

In [24]:
model = DynamicModel()
R1, P1, L1, W1 = model.transition(
    R=100, P=1775, L=500, W=500000,
    x=-550, xm=0,
    eps1=0.0, eps2=0.0, eps3=0.0
)
print(R1, P1, L1, W1)

138.46905894435938 1598.3902326251812 -50 531915.945421523
